In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import geopandas as gpd

df = pd.read_csv("../data/processed/marine_data_cleaned.csv")

fig, ax = plt.subplots(figsize=(14, 7))

# Set ocean background color
ax.set_facecolor('#a8c8e8')

# Draw world map background for geographic context
try:
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
except Exception:
    import geodatasets
    world = gpd.read_file(geodatasets.get_path('naturalearth.land'))
world.plot(ax=ax, color='#d4c9a8', edgecolor='#5a5a5a', linewidth=0.7, zorder=1)

ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)

# KDE heatmap overlay
xy = np.vstack([df["LONGITUDE"], df["LATITUDE"]])
kde = gaussian_kde(xy, bw_method=0.15)
xgrid = np.linspace(-180, 180, 400)
ygrid = np.linspace(-90, 90, 200)
Xg, Yg = np.meshgrid(xgrid, ygrid)
Z = kde(np.vstack([Xg.ravel(), Yg.ravel()])).reshape(Xg.shape)

heatmap = ax.contourf(Xg, Yg, Z, levels=20, cmap="YlOrRd", alpha=0.65, zorder=2)
plt.colorbar(heatmap, ax=ax, label="Sampling Density", shrink=0.6)

ax.scatter(df["LONGITUDE"], df["LATITUDE"], s=8, c="black", alpha=0.3, zorder=5)

ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)
ax.set_title("Geographic Distribution of BioTIME Marine Sites", fontsize=14)
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)

plt.tight_layout()
plt.savefig("biotime_site_map.png", dpi=150, bbox_inches="tight")
plt.show()
